In [72]:
# =========================================================
# CELL 1 - EVALUATION - PRECISION@K
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from datetime import datetime, timezone

from src.preprocessing.clean_text import clean_text
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming

from supabase import create_client, Client

print("✅ Import berhasil")

✅ Import berhasil


In [73]:
# =========================================================
# CELL 2 - KONEKSI SUPABASE
# =========================================================
SUPABASE_URL = os.getenv("SUPABASE_URL", "https://bnuzmrtiaciqlotxcgot.supabase.co")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
if not SUPABASE_KEY:
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "sb_publishable_Z8M8GISPVKMp1SGrQHrlLg_AZa8EOo-")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Koneksi Supabase berhasil")

✅ Koneksi Supabase berhasil


In [74]:
# =========================================================
# CELL 3 - HELPER UPSERT
# =========================================================
def to_records_safe(df: pd.DataFrame):
    return df.where(pd.notnull(df), None).to_dict(orient="records")

def upsert_batches(table_name: str, records: list, on_conflict: str, batch_size: int = 500):
    total = len(records)
    for i in range(0, total, batch_size):
        batch = records[i:i+batch_size]
        supabase.table(table_name).upsert(batch, on_conflict=on_conflict).execute()
    print(f"✅ Upsert ke {table_name}: {total} baris")

In [75]:
# =========================================================
# LOAD DATA
# =========================================================
base_dir  = os.path.abspath('..')
tfidf_dir = os.path.join(base_dir, 'data', 'tfidf')

with open(os.path.join(tfidf_dir, 'vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)

tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, 'tfidf_matrix.npz'))
doc_index    = pd.read_csv(os.path.join(base_dir, 'data', 'cleaned_papers.csv'))

stop_words = get_stopwords()

print(f'✅ Total dokumen : {len(doc_index)} artikel')
print(f'✅ Matriks TF-IDF: {tfidf_matrix.shape}')

✅ Total dokumen : 200 artikel
✅ Matriks TF-IDF: (200, 1718)


d:\Tugas Akhir\paperCi\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Tugas Akhir\paperCi\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [76]:
# =========================================================
# FUNGSI PREPROCESSING QUERY
# Sesuai batasan: stemming hanya untuk bahasa Indonesia
# =========================================================
def preprocess_query(query: str) -> str:
    text   = clean_text(query)
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    if all(token.isascii() for token in tokens):
        return ' '.join(tokens)
    
    tokens = stemming(tokens)
    return ' '.join(tokens)

In [77]:
# =========================================================
# SEARCH FUNCTION
# =========================================================
def search(query: str, top_k: int = 10) -> pd.DataFrame:
    processed = preprocess_query(query).strip()
    if not processed:
        return pd.DataFrame(columns=['id', 'title', 'authors', 'year', 'source', 'category', 'similarity_score', 'rank'])

    query_vec = vectorizer.transform([processed])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    ranked_idx = np.argsort(scores)[::-1]
    results = doc_index.iloc[ranked_idx][['id', 'title', 'authors', 'year', 'source', 'category']].copy()
    results['similarity_score'] = scores[ranked_idx]

    # filter relevan skor > 0 dulu
    results = results[results['similarity_score'] > 0].copy()

    # baru ambil top_k
    results = results.head(top_k).reset_index(drop=True)
    results['rank'] = range(1, len(results) + 1)

    return results

print('✅ Fungsi search siap')

✅ Fungsi search siap


In [78]:
# =========================================================
# QUERY UJI (10 query, 4 kategori)
# =========================================================
queries_eval = [
    {"query": "machine learning",    "kategori": "Machine Learning"},
    {"query": "deep learning",       "kategori": "Machine Learning"},
    {"query": "web development",     "kategori": "Web Development"},
    {"query": "web application",     "kategori": "Web Development"},
    {"query": "cyber security",      "kategori": "Cyber Security"},
    {"query": "network security",    "kategori": "Cyber Security"},
    {"query": "mobile application",  "kategori": "Mobile Application"},
    {"query": "android application", "kategori": "Mobile Application"},
    {"query": "data mining",         "kategori": "Machine Learning"},
    {"query": "software security",   "kategori": "Cyber Security"},
]

In [79]:
# =========================================================
# JALANKAN SEARCH SEMUA QUERY
# =========================================================
all_results    = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    q        = item['query']
    kategori = item['kategori']
    results  = search(q, top_k=10)
    all_results[q] = results

    for _, row in results.iterrows():
        kemunculan_all[row['title']] += 1
        kemunculan_kat[kategori][row['title']] += 1

print('✅ Semua query selesai diproses!')


✅ Semua query selesai diproses!


In [80]:
# =========================================================
# TEMPLATE GROUND TRUTH (ISI MANUAL)
# =========================================================
# Kolom 'relevan' isi manual: 1 relevan, 0 tidak relevan
rows = []
for item in queries_eval:
    q = item['query']
    kat = item['kategori']
    res = all_results[q]

    for _, r in res.iterrows():
        rows.append({
            'query': q,
            'kategori': kat,
            'rank': int(r['rank']),
            'article_id': int(r['id']),
            'title': r['title'],
            'similarity_score': float(r['similarity_score']),
            'relevan': np.nan  # isi manual nanti
        })

gt_template = pd.DataFrame(rows)

eval_dir = os.path.join(base_dir, 'data', 'evaluation')
os.makedirs(eval_dir, exist_ok=True)

template_path = os.path.join(eval_dir, 'ground_truth_template.csv')
gt_template.to_csv(template_path, index=False)

print(f'✅ Template ground truth dibuat: {template_path}')
gt_template.head(10)

✅ Template ground truth dibuat: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_template.csv


,query,kategori,rank,article_id,title,similarity_score,relevan
0,machine learning,Machine Learning,1,40,Machine learning and deep learning: A review o...,0.623236,NaN
1,machine learning,Machine Learning,2,39,When machine learning meets privacy: A survey ...,0.609889,NaN
2,machine learning,Machine Learning,3,7,Machine learning and deep learning: C. Janiesc...,0.599380,NaN
3,machine learning,Machine Learning,4,18,An overview of machine learning classification...,0.598438,NaN
4,machine learning,Machine Learning,5,38,Scientific machine learning benchmarks,0.563391,NaN
5,machine learning,Machine Learning,6,35,Financial applications of machine learning: A ...,0.558102,NaN
6,machine learning,Machine Learning,7,25,Machine learning in chemical engineering: A pe...,0.532695,NaN
7,machine learning,Machine Learning,8,8,Machine learning foundations,0.527485,NaN
8,machine learning,Machine Learning,9,23,Machine learning and applications in microbiology,0.523502,NaN
9,machine learning,Machine Learning,10,11,Using machine learning to detect misstatements,0.506476,NaN


In [81]:
# =========================================================
# BUAT FILE LABELED DARI TEMPLATE (SEMENTARA SEMUA 0)
# =========================================================
template_path = os.path.join(eval_dir, 'ground_truth_template.csv')
gt_labeled_path = os.path.join(eval_dir, 'ground_truth_labeled.csv')

tmp = pd.read_csv(template_path)
tmp['relevan'] = 0  # isi awal, nanti edit manual baris yang relevan jadi 1
tmp.to_csv(gt_labeled_path, index=False)

print(f'✅ File dibuat: {gt_labeled_path}')
tmp.head(10)

✅ File dibuat: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_labeled.csv


,query,kategori,rank,article_id,title,similarity_score,relevan
0,machine learning,Machine Learning,1,40,Machine learning and deep learning: A review o...,0.623236,0
1,machine learning,Machine Learning,2,39,When machine learning meets privacy: A survey ...,0.609889,0
2,machine learning,Machine Learning,3,7,Machine learning and deep learning: C. Janiesc...,0.599380,0
3,machine learning,Machine Learning,4,18,An overview of machine learning classification...,0.598438,0
4,machine learning,Machine Learning,5,38,Scientific machine learning benchmarks,0.563391,0
5,machine learning,Machine Learning,6,35,Financial applications of machine learning: A ...,0.558102,0
6,machine learning,Machine Learning,7,25,Machine learning in chemical engineering: A pe...,0.532695,0
7,machine learning,Machine Learning,8,8,Machine learning foundations,0.527485,0
8,machine learning,Machine Learning,9,23,Machine learning and applications in microbiology,0.523502,0
9,machine learning,Machine Learning,10,11,Using machine learning to detect misstatements,0.506476,0


In [91]:
# =========================================================
# LOAD / INIT GROUND TRUTH
# =========================================================
import os
import pandas as pd

gt_labeled_path = os.path.join(eval_dir, 'ground_truth_labeled.csv')
template_path = os.path.join(eval_dir, 'ground_truth_template.csv')

if os.path.exists(gt_labeled_path):
    gt_df = pd.read_csv(gt_labeled_path)
    print(f'✅ Pakai file labeled: {gt_labeled_path}')
elif os.path.exists(template_path):
    gt_df = pd.read_csv(template_path).copy()
    if 'relevan' not in gt_df.columns:
        gt_df['relevan'] = 0
    gt_df['relevan'] = gt_df['relevan'].fillna(0).astype(int)
    print('⚠️ ground_truth_labeled.csv belum ada, pakai template (default relevan=0)')
else:
    raise FileNotFoundError('Template ground truth belum dibuat. Jalankan cell pembuatan template dulu.')

required_cols = {'query', 'article_id', 'relevan'}
missing = required_cols - set(gt_df.columns)
if missing:
    raise ValueError(f'Kolom wajib ground truth kurang: {missing}')

gt_df['relevan'] = gt_df['relevan'].fillna(0).astype(int)
invalid = set(gt_df['relevan'].unique()) - {0, 1}
if invalid:
    raise ValueError(f'Nilai relevan harus 0/1. Ditemukan: {invalid}')

print('✅ gt_df siap dipakai')
gt_df.head(5)

✅ Pakai file labeled: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_labeled.csv
✅ gt_df siap dipakai


,query,kategori,rank,article_id,title,similarity_score,relevan
0,machine learning,Machine Learning,1,40,Machine learning and deep learning: A review o...,0.623236,1
1,machine learning,Machine Learning,2,39,When machine learning meets privacy: A survey ...,0.609889,1
2,machine learning,Machine Learning,3,7,Machine learning and deep learning: C. Janiesc...,0.599380,1
3,machine learning,Machine Learning,4,18,An overview of machine learning classification...,0.598438,1
4,machine learning,Machine Learning,5,38,Scientific machine learning benchmarks,0.563391,1


In [90]:
import os, pandas as pd
base_dir = os.path.abspath("..")
p = os.path.join(base_dir, "data", "evaluation", "ground_truth_labeled.csv")
df = pd.read_csv(p)

ones = {
    "machine learning": "ALL",
    "deep learning": [40,7,43,25,35,197,181,34,18],
    "web development": "ALL",
    "web application": [67,66,51,80,91,97],
    "cyber security": "ALL",
    "network security": [150,114,106,132,139,111,130,129,137,124],
    "mobile application": "ALL",
    "android application": [197,181,159,166,195,186,176],
    "data mining": [47,182],
    "software security": [106,132,139,111,130,129,137,124,149],
}

df["relevan"] = 0
df["query"] = df["query"].str.lower().str.strip()

for q, rule in ones.items():
    if rule == "ALL":
        df.loc[df["query"] == q, "relevan"] = 1
    else:
        df.loc[(df["query"] == q) & (df["article_id"].isin(rule)), "relevan"] = 1

df.to_csv(p, index=False)
print("updated:", p)
print(df.groupby("query")["relevan"].sum())

updated: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_labeled.csv
query
android application     7
cyber security         10
data mining             2
deep learning           9
machine learning       10
mobile application     10
network security       10
software security       9
web application         6
web development        10
Name: relevan, dtype: int64


In [92]:
# =========================================================
# TABEL 1 - HASIL PENCARIAN PER QUERY + LABEL
# =========================================================
print('\n' + '='*90)
print('TABEL 1 - HASIL PENCARIAN PER QUERY')
print('='*90)

tabel1_rows = []

for item in queries_eval:
    q = item['query']
    kat = item['kategori']
    res = all_results[q].copy()

    # merge label relevansi by query + article_id
    sub_gt = gt_df[gt_df['query'] == q][['article_id', 'relevan']].copy()
    sub_gt = sub_gt.rename(columns={'article_id': 'id'})
    merged = res.merge(sub_gt, on='id', how='left')
    merged['relevan'] = merged['relevan'].fillna(0).astype(int)

    print(f'\n🔍 Query    : "{q}"')
    print(f'📂 Kategori : {kat}')
    print(f'{"Rank":<5} {"Score":>8} {"Relevan":>8}  Judul')
    print('-'*90)

    for _, row in merged.iterrows():
        rel_txt = 'Ya' if row['relevan'] == 1 else 'Tidak'
        print(f'{int(row["rank"]):<5} {row["similarity_score"]:>8.4f} {rel_txt:>8}  {str(row["title"])[:60]}')

        tabel1_rows.append({
            'Query': q,
            'Kategori': kat,
            'Rank': int(row['rank']),
            'Article ID': int(row['id']),
            'Judul': row['title'],
            'Penulis': row['authors'],
            'Tahun': row['year'],
            'Similarity Score': round(float(row['similarity_score']), 4),
            'Relevan': rel_txt
        })

tabel1_df = pd.DataFrame(tabel1_rows)


TABEL 1 - HASIL PENCARIAN PER QUERY

🔍 Query    : "machine learning"
📂 Kategori : Machine Learning
Rank     Score  Relevan  Judul
------------------------------------------------------------------------------------------
1       0.6232       Ya  Machine learning and deep learning: A review of methods and 
2       0.6099       Ya  When machine learning meets privacy: A survey and outlook
3       0.5994       Ya  Machine learning and deep learning: C. Janiesch et al.
4       0.5984       Ya  An overview of machine learning classification techniques
5       0.5634       Ya  Scientific machine learning benchmarks
6       0.5581       Ya  Financial applications of machine learning: A literature rev
7       0.5327       Ya  Machine learning in chemical engineering: A perspective
8       0.5275       Ya  Machine learning foundations
9       0.5235       Ya  Machine learning and applications in microbiology
10      0.5065       Ya  Using machine learning to detect misstatements

🔍 Query    : 

In [93]:
# =========================================================
# TABEL 2 - PRECISION@K PER QUERY
# =========================================================
K_values = [5, 10]
eval_rows = []

for item in queries_eval:
    q = item['query']
    kat = item['kategori']
    res = all_results[q].copy()

    sub_gt = gt_df[gt_df['query'] == q][['article_id', 'relevan']].copy()
    sub_gt = sub_gt.rename(columns={'article_id': 'id'})
    merged = res.merge(sub_gt, on='id', how='left')
    merged['relevan'] = merged['relevan'].fillna(0).astype(int)

    row = {
        'Query': q,
        'Kategori': kat,
        'Retrieved': int(len(merged))
    }

    for k in K_values:
        topk = merged.head(k)
        rel_k = int(topk['relevan'].sum())
        p_k = rel_k / k  # definisi standard precision@k
        row[f'Relevan@{k}'] = rel_k
        row[f'P@{k}'] = round(p_k, 4)

    eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)

print('\n' + '='*90)
print('TABEL 2 - PRECISION@K PER QUERY')
print('='*90)
print(eval_df[['Query', 'Kategori', 'Retrieved', 'Relevan@5', 'P@5', 'Relevan@10', 'P@10']].to_string(index=False))

print('\n' + '-'*90)
for k in K_values:
    avg = eval_df[f'P@{k}'].mean()
    print(f'Mean P@{k}: {avg:.4f} ({avg*100:.2f}%)')

kat_avg = eval_df.groupby('Kategori', as_index=False)['P@10'].mean()
kat_avg.columns = ['Kategori', 'Rata-rata P@10']
kat_avg['Rata-rata P@10'] = kat_avg['Rata-rata P@10'].round(4)

print('\nRata-rata P@10 per kategori:')
print(kat_avg.to_string(index=False))


TABEL 2 - PRECISION@K PER QUERY
              Query           Kategori  Retrieved  Relevan@5  P@5  Relevan@10  P@10
   machine learning   Machine Learning         10          5  1.0          10   1.0
      deep learning   Machine Learning         10          5  1.0           9   0.9
    web development    Web Development         10          5  1.0          10   1.0
    web application    Web Development         10          2  0.4           6   0.6
     cyber security     Cyber Security         10          5  1.0          10   1.0
   network security     Cyber Security         10          5  1.0          10   1.0
 mobile application Mobile Application         10          5  1.0          10   1.0
android application Mobile Application         10          5  1.0           7   0.7
        data mining   Machine Learning         10          2  0.4           2   0.2
  software security     Cyber Security         10          4  0.8           9   0.9

------------------------------------------

In [94]:
# =========================================================
# TABEL 3 - JUMLAH KEMUNCULAN ARTIKEL
# =========================================================
print('\n' + '='*90)
print('TABEL 3A - JUMLAH KEMUNCULAN ARTIKEL (SEMUA QUERY)')
print('='*90)

kemunculan_df = pd.DataFrame([
    {'Judul Artikel': title[:80], 'Jumlah Kemunculan': count}
    for title, count in sorted(kemunculan_all.items(), key=lambda x: -x[1])
])

print(kemunculan_df.head(10).to_string(index=False))

print('\n' + '='*90)
print('TABEL 3B - JUMLAH KEMUNCULAN PER KATEGORI')
print('='*90)

tabel3b_rows = []
for kat in ['Machine Learning', 'Web Development', 'Cyber Security', 'Mobile Application']:
    if kat not in kemunculan_kat:
        continue

    data = kemunculan_kat[kat]
    top3 = sorted(data.items(), key=lambda x: -x[1])[:3]

    print(f'\n📂 {kat}')
    print(f'  {"Judul Artikel":<70} Kemunculan')
    print('  ' + '-'*84)

    for title, count in top3:
        print(f'  {str(title)[:70]:<70} {count}x')
        tabel3b_rows.append({
            'Kategori': kat,
            'Judul Artikel': title,
            'Jumlah Kemunculan': count
        })

tabel3b_df = pd.DataFrame(tabel3b_rows)


TABEL 3A - JUMLAH KEMUNCULAN ARTIKEL (SEMUA QUERY)
                                                                   Judul Artikel  Jumlah Kemunculan
A new mobile application of agricultural pests recognition using deep learning i                  4
Human monkeypox classification from skin lesion images with deep pre-trained net                  3
Deep learning methods for accurate skin cancer recognition and mobile applicatio                  3
Towards a new learning experience through a mobile application with augmented re                  3
User experience analysis on mobile application design using user experience ques                  3
                            A systematic literature review on the cyber security                  3
Analysis of cyber security knowledge gaps based on cyber security body of knowle                  3
                   The difference between cyber security vs information security                  3
A comprehensive review of cyber security vulnera

In [95]:
# Rata-rata keseluruhan
print('\n' + '-'*75)
for k in K_values:
    avg = eval_df[f'P@{k}'].mean()
    print(f'Mean Average P@{k}  : {avg:.4f}  ({avg*100:.1f}%)')

# Rata-rata per kategori
print('\nRata-rata P@10 per Kategori:')
kat_avg = eval_df.groupby('Kategori')['P@10'].mean().reset_index()
kat_avg.columns = ['Kategori', 'Rata-rata P@10']
kat_avg['Rata-rata P@10'] = kat_avg['Rata-rata P@10'].round(4)
print(kat_avg.to_string(index=False))



---------------------------------------------------------------------------
Mean Average P@5  : 0.8600  (86.0%)
Mean Average P@10  : 0.8300  (83.0%)

Rata-rata P@10 per Kategori:
          Kategori  Rata-rata P@10
    Cyber Security          0.9667
  Machine Learning          0.7000
Mobile Application          0.8500
   Web Development          0.8000


In [96]:
# =========================================================
# SIMPAN OUTPUT EVALUASI
# =========================================================
os.makedirs(eval_dir, exist_ok=True)

tabel1_df.to_csv(os.path.join(eval_dir, 'tabel1_hasil_pencarian.csv'), index=False)
eval_df.to_csv(os.path.join(eval_dir, 'tabel2_precision_at_k.csv'), index=False)
kat_avg.to_csv(os.path.join(eval_dir, 'tabel2_rata_rata_per_kategori.csv'), index=False)
kemunculan_df.to_csv(os.path.join(eval_dir, 'tabel3a_kemunculan_semua.csv'), index=False)
tabel3b_df.to_csv(os.path.join(eval_dir, 'tabel3b_kemunculan_per_kategori.csv'), index=False)

print('\n✅ Semua file tersimpan di data/evaluation/')
print(' - ground_truth_template.csv')
print(' - tabel1_hasil_pencarian.csv')
print(' - tabel2_precision_at_k.csv')
print(' - tabel2_rata_rata_per_kategori.csv')
print(' - tabel3a_kemunculan_semua.csv')
print(' - tabel3b_kemunculan_per_kategori.csv')


✅ Semua file tersimpan di data/evaluation/
 - ground_truth_template.csv
 - tabel1_hasil_pencarian.csv
 - tabel2_precision_at_k.csv
 - tabel2_rata_rata_per_kategori.csv
 - tabel3a_kemunculan_semua.csv
 - tabel3b_kemunculan_per_kategori.csv


In [97]:
# =========================================================
# CELL 17 - SIMPAN EVALUATION KE SUPABASE
# table: evaluation_precision_at_k
# kolom: compared_text, k, retrieved_count, relevant_retrieved, precision_at_k, updated_at
# PK: (compared_text, k)
# =========================================================
ts = datetime.now(timezone.utc).isoformat()

rows_eval_db = []

for _, r in eval_df.iterrows():
    q = r["Query"]

    rows_eval_db.append({
        "compared_text": q,
        "k": 5,
        "retrieved_count": int(r["Retrieved"]),
        "relevant_retrieved": int(r["Relevan@5"]),
        "precision_at_k": float(r["P@5"]),
        "updated_at": ts
    })

    rows_eval_db.append({
        "compared_text": q,
        "k": 10,
        "retrieved_count": int(r["Retrieved"]),
        "relevant_retrieved": int(r["Relevan@10"]),
        "precision_at_k": float(r["P@10"]),
        "updated_at": ts
    })

upsert_batches(
    table_name="evaluation_precision_at_k",
    records=rows_eval_db,
    on_conflict="compared_text,k",
    batch_size=500
)

✅ Upsert ke evaluation_precision_at_k: 20 baris


In [98]:
# =========================================================
# - VALIDASI HASIL EVALUATION DI SUPABASE
# =========================================================
check = supabase.table("evaluation_precision_at_k").select("compared_text,k,precision_at_k", count="exact").limit(20).execute()
print("✅ Total row evaluation_precision_at_k:", check.count)
check.data

✅ Total row evaluation_precision_at_k: 20


[{'compared_text': 'machine learning', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'machine learning', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'deep learning', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'deep learning', 'k': 10, 'precision_at_k': 0.9},
 {'compared_text': 'web development', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'web development', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'web application', 'k': 5, 'precision_at_k': 0.4},
 {'compared_text': 'web application', 'k': 10, 'precision_at_k': 0.6},
 {'compared_text': 'cyber security', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'cyber security', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'network security', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'network security', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'mobile application', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'mobile application', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'android app